# Chapitre 7 - Cours 2 : optimisme, Cp, AIC, BIC, nombre effectif de paramètres et dimension VC

Le notebook précédent a montré que l'erreur d'apprentissage $\overline{\mathrm{err}}$ est
trop optimiste. Nous quantifions ici cet **optimisme** (sections 7.4 à 7.9 du livre) et en
déduisons des critères analytiques de sélection de modèle :

1. erreur intra-échantillon, optimisme et formule $\omega = \frac2N\sum\mathrm{Cov}(\hat y_i, y_i)$ ;
2. les critères **Cp** et **AIC**, illustrés sur les données de phonèmes (figure 7.4) ;
3. le **nombre effectif de paramètres** $\mathrm{df}(\mathbf S) = \mathrm{trace}(\mathbf S)$ ;
4. le critère **BIC** et son interprétation bayésienne, comparé à AIC ;
5. la **dimension de Vapnik-Chervonenkis** (figures 7.5 et 7.6).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch07_selection as sel
set_style()

## 1. Erreur intra-échantillon et optimisme

Dans cette section, les entrées $x_1, \dots, x_N$ sont **fixées** et seules les réponses
sont aléatoires. On imagine de **nouvelles** réponses $Y_i^0$ observées aux mêmes points
$x_i$, indépendantes des $y_i$ mais de même loi. L'**erreur intra-échantillon**
(éq. (7.18) du livre) est

$$
\mathrm{Err}_{\mathrm{in}} = \frac1N \sum_{i=1}^N \mathbb E_{Y^0}\big[L(Y_i^0, \hat f(x_i)) \,\big|\, \mathcal T\big].
$$

Elle diffère de $\mathrm{Err}_{\mathcal T}$ uniquement par le lieu des points test : on
évalue aux points d'apprentissage au lieu de points $X^0$ nouveaux.

On définit l'**optimisme** comme l'écart $\mathrm{op} = \mathrm{Err}_{\mathrm{in}} - \overline{\mathrm{err}}$
(éq. (7.19)), et l'**optimisme moyen** $\omega = \mathbb E_{\mathbf y}(\mathrm{op})$ (éq. (7.20)),
où l'espérance porte sur les réponses d'apprentissage. Pour la perte quadratique (et
d'autres pertes), on a le résultat fondamental (éq. (7.21), démontré dans l'exercice 7.4) :

$$\omega = \frac2N \sum_{i=1}^N \mathrm{Cov}(\hat y_i, y_i).$$

Autrement dit,
$$\mathbb E_{\mathbf y}(\mathrm{Err}_{\mathrm{in}}) = \mathbb E_{\mathbf y}(\overline{\mathrm{err}}) + \frac2N\sum_{i=1}^N \mathrm{Cov}(\hat y_i, y_i). \tag{7.22}$$

> **Intuition.** Plus une observation $y_i$ influence sa propre prédiction $\hat y_i$,
> plus l'erreur d'apprentissage est flatteuse. Un modèle qui interpole ($\hat y_i = y_i$)
> a une covariance maximale $\mathrm{Var}(y_i) = \sigma^2_\varepsilon$ et une erreur
> d'apprentissage nulle.

**Cas linéaire.** Si $\hat{\mathbf y} = \mathbf H\mathbf y$ est un ajustement par
moindres carrés sur $d$ variables (ou fonctions de base) et $Y = f(X) + \varepsilon$ avec
$\mathrm{Var}(\varepsilon) = \sigma_\varepsilon^2$, alors
$\mathrm{Cov}(\hat{\mathbf y}, \mathbf y) = \mathbf H\,\mathrm{Var}(\mathbf y) = \sigma^2_\varepsilon\mathbf H$,
donc $\sum_i \mathrm{Cov}(\hat y_i, y_i) = \sigma^2_\varepsilon\,\mathrm{trace}(\mathbf H) = d\,\sigma^2_\varepsilon$
(éq. (7.23)), et

$$\mathbb E_{\mathbf y}(\mathrm{Err}_{\mathrm{in}}) = \mathbb E_{\mathbf y}(\overline{\mathrm{err}}) + 2\,\frac dN\,\sigma^2_\varepsilon. \tag{7.24}$$

L'optimisme croît **linéairement** avec $d$ et décroît comme $1/N$.

Vérifions (7.24) par simulation : $N = 40$ entrées fixées, $f(x) = \sin(2\pi x_1) + x_2$,
régression polynomiale en $x_1$ de degré variable (avec $x_2$), $\sigma_\varepsilon = 1$.
On estime $\mathbb E(\mathrm{Err}_{\mathrm{in}} - \overline{\mathrm{err}})$ en tirant
beaucoup de vecteurs $\mathbf y$ et, pour chacun, des réponses neuves $\mathbf Y^0$.

In [ ]:
rng = np.random.RandomState(0)
N, sig, n_rep = 40, 1.0, 4000
X = rng.rand(N, 2)
fx = np.sin(2 * np.pi * X[:, 0]) + X[:, 1]
degrees = np.arange(1, 11)
opt_mc, d_list = [], []
for deg in degrees:
    B = np.column_stack([X[:, 0] ** j for j in range(1, deg + 1)] + [X[:, 1]])
    H = sel.hat_matrix(B)
    d_list.append(np.trace(H))
    Y = fx + sig * rng.randn(n_rep, N)
    Y0 = fx + sig * rng.randn(n_rep, N)
    Yhat = Y @ H.T
    err = ((Y - Yhat) ** 2).mean(1)
    err_in = ((Y0 - Yhat) ** 2).mean(1)
    opt_mc.append((err_in - err).mean())
d_list = np.array(d_list)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(d_list, 2 * d_list * sig ** 2 / N, color=BLUE, label=r"formule (7.24) : $2d\sigma_\varepsilon^2/N$")
ax.plot(d_list, opt_mc, "o", color=ORANGE, label="optimisme moyen simulé")
ax.set_xlabel("Nombre de paramètres $d$")
ax.set_ylabel(r"$\omega = \mathbb{E}(\mathrm{Err}_{\mathrm{in}} - \overline{\mathrm{err}})$")
ax.set_title("Optimisme de l'erreur d'apprentissage (moindres carrés)")
ax.legend()
plt.show()